In [ ]:
!pip install sentence-transformers faiss-cpu transformers gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers accelerate sentencepiece

In [1]:
import pandas as pd

df = pd.read_csv("/content/cars_ds_final.csv")

df.head()

ParserError: Error tokenizing data. C error: EOF inside string starting at row 10393

In [ ]:
print(df.columns)

Index(['Unnamed: 0', 'Make', 'Model', 'Variant', 'Ex-Showroom_Price',
       'Displacement', 'Cylinders', 'Valves_Per_Cylinder', 'Drivetrain',
       'Cylinder_Configuration',
       ...
       'Leather_Wrapped_Steering', 'Automatic_Headlamps', 'Engine_Type',
       'ASR_/_Traction_Control', 'Cruise_Control', 'USB_Ports',
       'Heads-Up_Display', 'Welcome_Lights', 'Battery', 'Electric_Range'],
      dtype='object', length=141)


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en-v1.5")
#print("Embeddings created:", len(embeddings))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
texts = []

for _, row in df.iterrows():

    text = f"""
    Car Brand: {row.get('Make', '')}
    Car Model: {row.get('Model', '')}
    Variant: {row.get('Variant', '')}
    Price: {row.get('Ex-Showroom_Price', '')}
    Fuel Type: {row.get('Fuel_Type', '')}
    Body Type: {row.get('Body_Type', '')}
    Mileage: {row.get('City_Mileage', '')}
    Transmission: {row.get('Transmission', '')}
    Seating Capacity: {row.get('Seating_Capacity', '')}
    """

    texts.append(text)

In [ ]:
embeddings = model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

embedding_array = np.array(embeddings).astype('float32')

index = faiss.IndexFlatL2(embedding_array.shape[1])

index.add(embedding_array)
print("FAISS index created with", index.ntotal, "entries")

FAISS index created with 1276 entries


In [ ]:
def retrieve(query, k=5):

    query_embedding = model.encode([query]).astype('float32')

    distances, indices = index.search(query_embedding, k)

    results = []

    for idx in indices[0]:
        results.append(texts[idx])

    return results

In [ ]:
retrieve("best mileage car under 10 lakh")

['\n    Car Brand: Mahindra\n    Car Model: Tuv300\n    Variant: T10\n    Price: Rs. 9,99,614\n    Fuel Type: Diesel\n    Body Type: SUV\n    Mileage: 18.49 km/litre\n    Transmission: \n    Seating Capacity: 7.0\n    ',
 '\n    Car Brand: Hyundai\n    Car Model: Grand I10\n    Variant: 1.2 Kappa Vtvt Magna At\n    Price: Rs. 6,52,328\n    Fuel Type: Petrol\n    Body Type: Hatchback\n    Mileage: 18.9 km/litre\n    Transmission: \n    Seating Capacity: 5.0\n    ',
 '\n    Car Brand: Hyundai\n    Car Model: Grand I10\n    Variant: 1.2 Kappa Vtvt Sportz At\n    Price: Rs. 7,05,538\n    Fuel Type: Petrol\n    Body Type: Hatchback\n    Mileage: 18.9 km/litre\n    Transmission: \n    Seating Capacity: 5.0\n    ',
 '\n    Car Brand: Mahindra\n    Car Model: Tuv300\n    Variant: T10 (O)\n    Price: Rs. 10,31,943\n    Fuel Type: Diesel\n    Body Type: SUV\n    Mileage: 18.49 km/litre\n    Transmission: \n    Seating Capacity: 7.0\n    ',
 '\n    Car Brand: Hyundai\n    Car Model: Grand I10\n  

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [ ]:
def chatbot(query):

    try:

        retrieved_docs = retrieve(query, k=3)

        if not retrieved_docs:
            return "Sorry, I could not find matching cars."

        answer = "## Recommended Cars\n\n"

        for doc in retrieved_docs:

            lines = doc.split("\n")

            brand = "Unknown"
            model_name = "Unknown"
            price = "N/A"
            fuel = "N/A"
            mileage = "N/A"
            body = "N/A"

            for line in lines:

                if "Car Brand:" in line:
                    brand = line.replace("Car Brand:", "").strip()

                elif "Car Model:" in line:
                    model_name = line.replace("Car Model:", "").strip()

                elif "Price:" in line:
                    price = line.replace("Price:", "").strip()

                elif "Fuel Type:" in line:
                    fuel = line.replace("Fuel Type:", "").strip()

                elif "Mileage:" in line:
                    mileage = line.replace("Mileage:", "").strip()

                elif "Body Type:" in line:
                    body = line.replace("Body Type:", "").strip()

            answer += f"""
🚗 {brand} {model_name}

💰 Price: {price}
⛽ Fuel: {fuel}
🛣️ Mileage: {mileage}
🚘 Body Type: {body}

-------------------

"""

        return answer

    except Exception as e:

        return f"Error occurred: {str(e)}"

In [ ]:
at

In [ ]:
while True:

    query = input("Ask about cars: ")

    if query.lower() == "exit":
        break

    response = chatbot(query)

    print("\nAnswer:\n")
    print(response)
    print("\n" + "="*60 + "\n")


Answer:

## Recommended Cars


🚗 Hyundai Grand I10

💰 Price: Rs. 6,52,328
⛽ Fuel: Petrol
🛣️ Mileage: 18.9 km/litre
🚘 Body Type: Hatchback

-------------------


🚗 Hyundai Grand I10

💰 Price: Rs. 7,05,538
⛽ Fuel: Petrol
🛣️ Mileage: 18.9 km/litre
🚘 Body Type: Hatchback

-------------------


🚗 Hyundai Grand I10

💰 Price: Rs. 6,20,637
⛽ Fuel: Petrol
🛣️ Mileage: 18.9 km/litre
🚘 Body Type: Hatchback

-------------------





Answer:

## Recommended Cars


🚗 Hyundai Creta

💰 Price: Rs. 9,99,990
⛽ Fuel: Diesel
🛣️ Mileage: 21.38 km/litre
🚘 Body Type: SUV

-------------------


🚗 Hyundai Creta

💰 Price: Rs. 13,36,033
⛽ Fuel: Diesel
🛣️ Mileage: 17.01 km/litre
🚘 Body Type: SUV

-------------------


🚗 Hyundai Creta

💰 Price: Rs. 11,07,167
⛽ Fuel: Diesel
🛣️ Mileage: 21.38 km/litre
🚘 Body Type: SUV

-------------------




